In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ranksums
import json

In [ ]:
expts = {'XLI2':'Schisto',
         'XCE1':'CoV',
         'JYH3':'BoNT/A',
         'final':'Test'}
expt_order = ['XLI2','JYH3','XCE1','final']
alg_order = [('nanomap', 'individual'),
             ('nanomap', 'meta'),
             ('scoper', 'individual'),
             ('mmseqs', 'individual')]

scores = pd.read_csv('all_scores.csv',index_col=0)
scores

In [ ]:
stability = scores.groupby(['method','name','expt','kind'])['ari_to_full'].min().reset_index().rename({'ari_to_full':'stability'},axis=1)
scores = scores.merge(stability,on=['method','name','expt','kind'],how='left')
scores

In [ ]:
score_order = ['silhouette','ranksum_score','ari','stability']

summary = []
best_settings = {}
for alg,group in scores[scores['factor']==1].groupby(['method','kind']):
    for expt in expt_order[:-1]: #leave out test set
        other_expts = [e for e in expt_order if e not in {expt, 'final'}] #leave out test set
        other_scores = group.pivot(index='name',columns='expt',values=score_order)[:][other_expts]
        best_name = (other_scores*(other_scores>0)).prod(axis=1).idxmax()
        best_settings[f'{alg[0]}-{alg[1]}-{expt}'] = best_name
        
        best_scores = []
        for sc in score_order:
            best_scores.append(group.loc[(group['name']==best_name)&(group['expt']==expt),sc].values[0]) #score for overall best params
        summary.append([alg, expts[expt]]+best_scores)

summary = pd.DataFrame(summary,columns=['method','expt']+score_order)
summary

In [ ]:
json.dump(best_settings,open('best_settings.json','w'),indent=2)
best_settings

In [ ]:
# add test set evaluation
test_summary = []
for alg,group in scores[scores['factor']==1].groupby(['method','kind']):
    best = best_settings[f'{alg[0]}-{alg[1]}-JYH3'] #use JYH3 because that one always picks the settings that are in the majority
    row = scores[(scores['factor']==1)&(scores['expt']=='final')&(scores['method']==alg[0])&(scores['kind']==alg[1])&(scores['name']==best)]
    test_summary.append([alg]+row.iloc[0,:][score_order].tolist())

test_summary = pd.DataFrame(test_summary,columns=['method']+score_order)
test_summary['expt'] = 'Test'

summary = pd.concat([summary,test_summary],axis=0).reset_index(drop=True)
summary

In [ ]:
for expt,group in summary.groupby('expt'):
    summary.loc[group.index,'norm_ranksum_score'] = group['ranksum_score']/group.loc[group['method'].map(lambda x: (x[1]=='individual') and (x[0]=='nanomap')),'ranksum_score'].values[0]

In [ ]:
colorblind = sns.color_palette("colorblind")
tab10 = plt.get_cmap('tab10')
letters = 'ABCD'

score_order = ['silhouette','norm_ranksum_score','ari','stability']

name_convert = {
    'ari':'ARI',
    'silhouette':'Silhouette',
    'norm_ranksum_score':'Normalized\nPhenotypic Quality',
    'stability':'Stability'
}

color_id = {
    ('nanomap', 'individual'):9,
    ('nanomap', 'meta'):0,
    ('scoper', 'individual'):1,
    ('mmseqs', 'individual'):2
}

alg_name = {
    ('nanomap', 'individual'):'NanoMAP (ind)',
    ('nanomap', 'meta'):'NanoMAP (meta)',
    ('scoper', 'individual'):'SCOPer',
    ('mmseqs', 'individual'):'MMseqs2'
}

summary['alg_name'] = summary['method'].map(lambda alg: alg_name[alg])

plt.figure(figsize=[12,8])
for i,sc in enumerate(score_order):

    if sc == 'stability':
        for_plot = summary[summary['expt']!='Test']
    else:
        for_plot = summary
    
    ax = plt.subplot(2,2,i+1)
    sns.barplot(data=for_plot, x='expt', y=sc, hue='alg_name',
                hue_order=[alg_name[alg] for alg in alg_order], legend=(i==1), palette=[colorblind[color_id[x]] for x in alg_order])

    for l, label in enumerate(ax.get_xticklabels()):
        if l == 3:
            label.set_color('gray')
        else:
            label.set_color(tab10(l+3))

    if sc=='silhouette':
        plt.yticks([0,0.1,0.2,0.3,0.4])
    
    if i==1:
        plt.legend(loc=[0.04,0.05],fontsize=10)

    plt.xlabel('')

    plt.ylabel(name_convert[sc],fontsize=16)

    plt.xticks(fontsize=16,fontweight='bold')
    plt.yticks(fontsize=16)

    ax.text(-0.15, 1.05, letters[i], fontsize=32, transform=ax.transAxes)
plt.subplots_adjust(hspace=0.35,wspace=0.35)
plt.savefig('fig3.png',dpi=300,bbox_inches='tight')